# Stefan problem Description

Testcase used in the following work:
L. Wei, G. Bois, V. Pandeya, V. S. Nikolayev. (2025). _Multiscale simulations on the influence of contact line evaporation during nucleate boiling_. Applied Thermal Engineering. 

## Objective
This (quasi) 1D test case is widely used for the verification of phase-change models, specifically focusing on the Stefan problem, which involves phase change driven by temperature gradients.

## Remark
In the Stefan problem, a temperature gradient within the **vapor** region induces evaporation, which in turn drives the interface velocity. The governing equations are solved on the **vapor side only**.

## Summary of Initial and Boundary Conditions
- **Boundary Conditions (BCs):**
  - **Dynamics**: 
    - Free-slip condition on the top and bottom walls.
    - Fixed wall on the left side.
    - Outlet condition on the right side.
  - **Thermal**: 
    - Adiabatic conditions on the top and bottom walls.
    - Fixed temperature at $ T_w $ on the left wall.
    - Fixed temperature at $ T_{\text{sat}} $ on the right boundary.
  - **Phase**: All boundaries set to symmetry.

- **Initial Conditions (ICs):**
  - **Dynamics**: Initial velocity is zero throughout the domain.
  - **Thermal**: A linear temperature gradient is established within the vapor, ranging from $ T_w $ at $ x=0 $ to $ T_{\text{sat}} $ at $ x=x_0 $. The liquid is initially at a uniform temperature of $ T_{\text{sat}} $.
  - **Phase**: Vapor extends from $ x=0 $ to $ x=x_0 $.

![ICs and BCs for Stefan Problem](src/figs/stefan.png)

## Case Specifications
- **Domain Length**: $ L = 1 \, \text{mm} $
- **Initial Interface Position**: $ x_i = 0.05 \, \text{mm} $ at $ t = 0 \, \text{s} $
- **Material Properties**: Properties for water under atmospheric conditions:

|               | $\rho$ $kg/m^3$| $\mu$ $Pa\cdot s$        | $\lambda$ $W/(m\cdot K)$ | $C_p$ $J/(kg\cdot K)$   |
|---------------|----------------|--------------------------|--------------------------|-------------------------|
| **Liquid**    | 958.37         | $2.8 \times 10^{-4}$     | 0.679                    | $4.21 \times 10^3$      |
| **Vapor**     | 0.597          | $1.227 \times 10^{-5}$   | 0.025                    | $2.077 \times 10^3$     |
|               |                |                          |                          |                         |
| $\sigma = 0.0589$ N/m, $\mathcal{L} = 2.256 \times 10^6$ J/kg                                                  | 

---

## Analytical Solution

The analytical solution provides the theoretical interface position $ x_i(t) $ and the temperature distribution $ \Delta T(x, t) $ within the vapor.

$$
x_i(t) = 2 \beta \sqrt{\alpha_v t}
$$

$$
\Delta T(x, t) = \Delta T + \left( \frac{-\Delta T}{\operatorname{erf}(\beta)} \right) \operatorname{erf}\left( \frac{x}{2 \sqrt{\alpha_v t}} \right)
$$

where:
$\alpha_v$ is the thermal diffusivity of the vapor, defined by $$\alpha_v=\frac{\lambda_v}{\rho_v cp_{v}} $$; 

$cp_v$ is the specific heat capacity of the vapor (J/kg·K); $\operatorname{erf}(\beta)$ is the error function; $\beta$ is determined by solving the transcendental equation:

$$
\beta \exp(\beta^2) \operatorname{erf}(\beta) = \frac{cp_{v} \Delta T}{\sqrt{\pi \mathcal{L}}}
$$


## Moduels for plot,  and funcitons for analytical solutions

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import math
from math import sqrt, pi
Mar = ['o','x','s','d','*','h','D','+','v','^','>','<','.','1','8','p','$f$','2','4']
colors = ['red', 'green', 'blue', 'orange', 'purple']

try:
    import medcoupling as mc  # Replace 'some_module' with the name of the module you want to load
    print("medcoupling loaded successfully!")
except ImportError as e:
    print("Failed to load module:", e)
    try:
        import scienceplots
        plt.style.use('science')
    except ImportError as e:
        print("Failed to load module:", e)
import os
current_directory = os.getcwd() 
from scipy import special
from scipy.optimize import fsolve

In [ ]:
Cp_v = 2.077e3   # Specific heat capacity of vapor (J/kg·K)
lambda_v = 0.025  # Thermal conductivity of vapor (W/m·K)
rho_v = 0.597     # Density of vapor (kg/m^3)
h_lg = 2.256e6

def sol_stefan(beta, dT):
    # beta = np.clip(beta, 1.e-4, 2.5)  # Constraining beta to avoid overflow in exp
    return beta * np.exp(beta**2) * special.erf(beta) - Cp_v * dT / (h_lg * sqrt(pi))
def temp(t, x, dT, beta, tshift):
    return dT + (-dT/special.erf(beta))*special.erf(x/2./sqrt(lambda_v / (rho_v * Cp_v)* (t+tshift)))
    # return 2.0 * beta * np.sqrt(lambda_v / (rho_v * Cp_v) * (t+tshift))
def x_position(t, beta,tshift):
    return 2.0 * beta * np.sqrt(lambda_v / (rho_v * Cp_v) * (t+tshift))
def velo_i(t, beta, tshift):
    return beta * np.sqrt(lambda_v / (rho_v * Cp_v) / (t+tshift))

# RUN setup

## Get Beta and $t_{\mathrm{ref}}$ (corresponding the initial interface position)

In [ ]:
## domain size and initial interface position:
xi_0 = 5.e-5  # initial interface position
L = 1.e-3
W = 1.e-4

## primary variable: dT and nb_noeuds
dT = np.asarray([1, 1, 1, 500, 5000])*10
level = np.asarray([0, 1, 2, 2, 2])
nombre_de_noeuds = int((L/xi_0))*2**level+1

In [ ]:
# Determine the range of beta for beta_guess
beta_values = np.linspace(0, 10, 100)  # Range of beta values
for i in range(len(dT)):
    plt.plot(beta_values, sol_stefan(beta_values, dT[i]), label=f'dT={dT[i]}')
plt.axhline(0, color='red', linestyle='--', label='y = 0')
plt.title('Plot of sol_stefan(beta)')
plt.xlabel('beta')
plt.ylabel('sol_stefan(beta)')
plt.xlim([0, 2.5])
plt.ylim([-0.01, 0.01])
# plt.xscale('log')
plt.grid()
plt.legend()
plt.show()

From the previous figure, an appropriate estimate for beta can be determined. Without this estimate, the beta_solution may fail to converge.

In [ ]:
beta_guess = [0.1, 0.1, 0.1,1, 1.5]
beta = []
# Solve the equation
for i in range(len(dT)):
    beta_solution = fsolve(sol_stefan, beta_guess[i], args=(dT[i]))
    # Verify convergence
    print('Residul:', sol_stefan(beta_solution, dT[i]))  # should print close to 0 if solution is found
    print(f"The value of beta is: {beta_solution[0]}")
    beta.append(beta_solution[0])

# Run set

In [ ]:
from trustutils import run

run.introduction("L. WEI","09/10/2024")
# Declaration of the TRUST version
run.TRUST_parameters("1.9.3")

In [ ]:

tshift = []   # corresponding reference time for different cases
for i in range(len(beta)):
    tshift_tmp = (xi_0/2./beta[i])**2*rho_v*Cp_v/lambda_v
    tshift.append(tshift_tmp)

tmax = np.asarray(tshift)*74.
dt_post = tmax/20.
nb_test = len(dT)

In [ ]:
from math import sqrt, pi, floor, log10
import numpy as np
# dt_max = [dt_popinet(N-1) for N in nombre_de_noeuds ]
dt_max = tmax/1000

# nombre_de_noeuds = int((L/xi_0)*4.+1)



run.reset()
nbprocs = 1
for y in range(nb_test) :
# for y in range(1) :
    fname =f'M{y}'
    name = f'dT{(dT[y])}_{nombre_de_noeuds[y]-1}cells'
    substitutions_dict = {
                          "nbn" : str(nombre_de_noeuds[y]),
                          "mdtmax" :str(dt_max[y]),
                          "tsimu" :str(tmax[y]),
                          "xmax" :str(L),
                          "ymax" :str(W),
                          "dtpost" :str(dt_post[y]),
                          "dT" :str(dT[y]),
                          "xinit" :str(xi_0)
                          }

    tc = run.addCaseFromTemplate("stefan_CL_paper.data"
                             ,targetDirectory=f"{fname}"
                             ,dic=substitutions_dict
                             ,nbProcs= nbprocs
                             ,targetData=f"{name}.data")
    if nbprocs > 1:
        tc.partition()
run.printCases()


In [ ]:
delta_x = L/(nombre_de_noeuds-1)*1.e6

# Print the table header
header = f"| {'Case':<13} | {'dX ($\\mu m$)':<20} | {'dT (K)':<10} | {'t_ref (ms)':<12} |"
separator = "-" * len(header)
print(separator)
print(header)
print(separator)

# Print each row of the table
for i in range(nb_test):
    print(f"| M{i:<12} | {delta_x[i]:<20.1f} | {dT[i]:<10} | {tshift[i]*1.e3:<12.3f} |")

print(separator)

In [ ]:
# Run all cases
# run.runCases()
run.runCases(verbose=False, preventConcurrent=True)

In [ ]:
## Performances calculs

run.tablePerf()

# Postprocessing

## Pre-load module

In [ ]:
def get_prb_data(file, time):
    from trustutils.files import SonSEGFile
 
    import numpy as np
    
    
    son_file = f'build/{file}'
    donne = SonSEGFile(son_file,None)
    compo = 0
    ncompo = donne.getnCompo()
    entries = donne.getEntries()
    
    # y_label = entries[compo].split()[0]
    # x_label = donne.getXLabel()
    
    # print("x_label : ", x_label)
    # print("y_label : ", y_label)
    
    # start, end = donne.getXTremePoints()
    # print("dom xmin and ymin : ", start)
    # print("dom xmax and ymax : ", end)
    
    t = donne.getValues(entries[0])[0]

    ## find closest value of t
    if time == None:
        idx = -1
    else:
        idx = (np.abs(t - time)).argmin()
        
    X = donne.getXAxis()
    
    Y = []
    for i in entries[compo::ncompo]:
        Y.append(list(donne.getValues(i)[1])[idx])
    if X[0] != X.min():
        X = X[::-1]
        Y = Y[::-1]
    return X, np.asarray(Y)
def get_position(fname, name, W):
    import numpy as np
    import os
    os.system(f'grep "^Volume_phase_0" build/{fname}/{name}.err > build/{fname}/vol.txt')
     
    time = []
    vol = []  
    with open(f'build/{fname}/vol.txt', 'r') as file:
    # Lire chaque ligne du fichier
        for line in file:
            # Séparer les valeurs de la ligne en utilisant l'espace comme délimiteur
            valeurs = line.split()
            # Ajouter la valeur de la deuxième colonne à la liste colonne_2
            vol.append(float(valeurs[1]))
            # Ajouter la valeur de la quatrième colonne à la liste colonne_4
            time.append(float(valeurs[3]))
        
    posi = np.array(vol) / W
    return np.asarray(time), posi

## Mesh convergence

### Temperature along x

In [ ]:
labs= ['','500','5000']

num_mar = 0
# for y in range(nb_test) :
for y in range(3) :
    fname =f'M{y}'
    name = f'dT{(dT[y])}_{nombre_de_noeuds[y]-1}cells'
    X, Y = get_prb_data(f'{fname}/{name}_T_ABSCISSE.son', tmax[y])
    freq = int(len(X)/20)
    plt.scatter(X[::freq]/xi_0/sqrt(74),Y[::freq]/dT[y]
             ,  marker=Mar[num_mar%len(Mar)]
             # ,  markevery= 60
         , color=colors[(num_mar//1)%len(colors)]
         ,  label=r'$\Delta x=$' +f'{2**(-y)*xi_0*1.e6:.1f}'+ r'$\mathrm{\mu m}$'
        )
    num_mar = num_mar +1

y = 0
x = np.linspace(0, L, 101)
T_ana = temp(tmax[y], x, dT[y], beta[y], tshift[y])
plt.plot(x/xi_0/sqrt(74),T_ana/dT[y]
         , color=colors[(num_mar//1)%len(colors)]
         ,  label=labs[y]+r'$Ste_{\mathrm{ref}}$, '+'Ana.'
        )
num_mar = num_mar +1 

    
plt.xlabel(r'$x/x_{i}$')
plt.ylabel(r"$(T-T_\mathrm{sat})/\Delta T$")                  
plt.legend(loc="best")
# plt.legend(loc="upper center", ncol=2, bbox_to_anchor=(0.5, 1.4))
plt.xlim(0, 1)
plt.ylim(0, 1)
# plt.title(r"$\frac{r}{R_0}$",r"$\frac{T_l-T_{sat}}{\Delta T_{\infty}}$")
# plt.savefig(f'{current_directory}/radius_tcl4bis.png', bbox_inches='tight')
# plt.savefig(f'Comparison_T_ana.eps', bbox_inches='tight')
plt.show()

### Interface position

In [ ]:
# Main plot
fig, ax = plt.subplots()
num_mar = 0
for y in range(3) :   
    fname =f'M{y}'
    name = f'dT{(dT[y])}_{nombre_de_noeuds[y]-1}cells'
    
    time, position = get_position(fname,name, W)
    # freq = int(len(time)/20)+2*y
    freq =1
    ax.plot(time[::freq]/tshift[y]+1,position[::freq]/xi_0
             ,  marker=Mar[num_mar%len(Mar)]
            ,  markevery=int(len(time)/20)+y*200
         , color=colors[(num_mar//2)%len(colors)]
         ,  label=r'$\Delta x=$' +f'{2**(-y)*xi_0*1.e6:.1f}'+ r'$\mathrm{\mu m}$'
        )
    num_mar = num_mar +1
y= 0
time = np.linspace(0, tmax[y], 101)
x_ana = x_position(time, beta[y], tshift[y])
plt.plot(time/tshift[y]+1,x_ana/xi_0
         , color=colors[(num_mar//2)%len(colors)]
         ,  label='Ana.'
        )

num_mar = num_mar +1

from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset

inset_ax= inset_axes(
    ax, width="40%", height="40%", 
    loc="lower right",
    bbox_to_anchor=(-0.05, 0.125, 1, 1),
    bbox_transform=ax.transAxes, borderpad=0,
)
num_mar = 0
for y in range(3) :    
    fname =f'M{y}'
    name = f'dT{(dT[y])}_{nombre_de_noeuds[y]-1}cells'
    
    time, position = get_position(fname,name, W)
    # freq = int(len(time)/20)+2*y
    freq =1
    inset_ax.plot(time[::freq]/tshift[y]+1,position[::freq]/xi_0
             ,  marker=Mar[num_mar%len(Mar)]
            ,  markevery=int(len(time)/2000)
         , color=colors[(num_mar//2)%len(colors)]
         ,  label=r'$\Delta x=$' +f'{2**(-y)*xi_0*1.e6:.1f}'+ r'$\mathrm{\mu m}$'
        )
    num_mar = num_mar +1
y= 0
time = np.linspace(0, tmax[y], 101)
x_ana = x_position(time, beta[y], tshift[y])
inset_ax.plot(time/tshift[y]+1,x_ana/xi_0
         , color=colors[(num_mar//2)%len(colors)]
         ,  label='Ana.'
        )
num_mar = num_mar +1

inset_ax.set_xlim(40, 40.2)  # Zoom region
inset_ax.set_ylim(6.33, 6.34)  # Zoom region

# Indicate zoomed region on the main plot
rect = plt.Rectangle((40, 6.325), 0.2, 0.022, edgecolor='red', facecolor='none', lw=2)
ax.add_patch(rect)
mark_inset(ax, inset_ax, loc1=3, loc2=1, fc="none", ec="0.5")  # Connect the zoomed region


ax.set_xlabel(r'$t/t_{\mathrm{ref}}$')
ax.set_ylabel(r"$x_i/x_{i, int}$")                  
ax.legend(loc="upper center", ncol=2, bbox_to_anchor=(0.5, 1.3))
# plt.title(r"$\frac{r}{R_0}$",r"$\frac{T_l-T_{sat}}{\Delta T_{\infty}}$")
# plt.savefig(f'{current_directory}/radius_tcl4bis.png', bbox_inches='tight')
plt.savefig(f'Mesh_convergence_xi_zoom.eps', bbox_inches='tight')
plt.show()



### Error of interface position at $74t_{\mathrm{ref}}$

In [ ]:
err = []
for y in range(3) :
    fname =f'M{y}'
    with open(f'build/{fname}/vol.txt', 'r') as file:
        last_line = file.readlines()[-1].strip()
        valeurs = last_line.split()
        vol = float(valeurs[1])
        time= float(valeurs[3])
        x_simu = np.asarray(vol)/W
        x_theric = x_position(tmax[y], beta[y], tshift[y])
        err.append(abs(x_simu-x_theric)/x_theric)


plt.scatter(nombre_de_noeuds[:3]-1, err, marker="o", label = f'present')
x = np.asarray(nombre_de_noeuds)
# plt.plot(x, 1/x**2, label='2nd order')  
plt.plot(x, 0.02/x, label='1st order')

plt.xlabel(r"Number of cells")
plt.ylabel(r"Relative error")    
plt.xscale('log', base=2)
plt.yscale('log')
plt.legend(loc="best")
# plt.title(r"$\frac{r}{R_0}$",r"$\frac{T_l-T_{sat}}{\Delta T_{\infty}}$")
# plt.savefig(f'{current_directory}/radius_tcl4bis.png', bbox_inches='tight')
# plt.savefig(f'{save_path}/radius_tcl4bis.eps', bbox_inches='tight')
plt.show()

## Effect of Stefan Number

### Temperature along x

In [ ]:
labs= ['', '', '','500','5000']

num_mar = 0
# for y in range(nb_test) :
for y in range(2, 5) :
    fname =f'M{y}'
    name = f'dT{(dT[y])}_{nombre_de_noeuds[y]-1}cells'
    X, Y = get_prb_data(f'{fname}/{name}_T_ABSCISSE.son', tmax[y])
    freq = int(len(X)/20)
    plt.scatter(X[::freq]/xi_0/sqrt(74),Y[::freq]/dT[y]
             ,  marker=Mar[num_mar%len(Mar)]
             # ,  markevery= 60
         , color=colors[(num_mar//1)%len(colors)]
         ,  label=labs[y]+r'$Ste_{\mathrm{ref}}$, '+'TrioCFD'
        )
    num_mar = num_mar +1

num_mar = 0
for y in range(2, 5) :  
    x = np.linspace(0, L, 101)
    T_ana = temp(tmax[y], x, dT[y], beta[y], tshift[y])
    plt.plot(x/xi_0/sqrt(74),T_ana/dT[y]
             , color=colors[(num_mar//1)%len(colors)]
             ,  label=labs[y]+r'$Ste_{\mathrm{ref}}$, '+'Ana.'
            )
    num_mar = num_mar +1
    
plt.xlabel(r'$x/x_{i}$')
plt.ylabel(r"$(T-T_\mathrm{sat})/\Delta T$")                  
# plt.legend(loc="best")
plt.legend(loc="upper center", ncol=2, bbox_to_anchor=(0.5, 1.4))
plt.xlim(0, 1)
plt.ylim(0, 1)
# plt.title(r"$\frac{r}{R_0}$",r"$\frac{T_l-T_{sat}}{\Delta T_{\infty}}$")
# plt.savefig(f'{current_directory}/radius_tcl4bis.png', bbox_inches='tight')
plt.savefig(f'Effect_Ste_T_ana.eps', bbox_inches='tight')
plt.show()

### Interface position

In [ ]:
labs= ['', '', '','500','5000']

num_mar = 0
for y in range(2,5) :
    fname =f'M{y}'
    name = f'dT{(dT[y])}_{nombre_de_noeuds[y]-1}cells'
    
    time, position = get_position(fname,name, W)
    freq = int(len(time)/20)
    plt.scatter(time[::freq]/tshift[y]+1,position[::freq]/xi_0
             ,  marker=Mar[num_mar%len(Mar)]
            #  ,  markevery=6000
         , color=colors[(num_mar//1)%len(colors)]
         ,  label=labs[y]+r'$Ste_{\mathrm{ref}}$, '+'TrioCFD'
        )
    num_mar = num_mar +1

num_mar = 0
# for y in range(nb_test) :
for y in range(2,5) :
# for y in [1] :
    
    time = np.linspace(0, tmax[y], 101)
    x_ana = x_position(time, beta[y], tshift[y])
    plt.plot(time/tshift[y]+1,x_ana/xi_0
             , color=colors[(num_mar//1)%len(colors)]
             ,  label=labs[y]+r'$Ste_{\mathrm{ref}}$, '+'Ana.'
            )
    
    num_mar = num_mar +1

plt.xlabel(r'$t/t_{\mathrm{ref}}$')
plt.ylabel(r"$x_i/x_{i, int}$")                  
plt.legend(loc="best")
# plt.xlim(0, 100)
plt.legend(loc="upper center", ncol=2, bbox_to_anchor=(0.5, 1.4))
# plt.ylim(0, 1)
# plt.title(r"$\frac{r}{R_0}$",r"$\frac{T_l-T_{sat}}{\Delta T_{\infty}}$")
# plt.savefig(f'{current_directory}/radius_tcl4bis.png', bbox_inches='tight')
plt.savefig(f'Effect_Ste_xi_ana.eps', bbox_inches='tight')
plt.show()

### Zoomed view

In [ ]:
num_mar = 0
for y in range(2,5) :
    fname =f'M{y}'
    name = f'dT{(dT[y])}_{nombre_de_noeuds[y]-1}cells'
    
    time, position = get_position(fname,name, W)
    freq = int(len(time)/20)
    plt.scatter(time[::freq]/tshift[y]+1,position[::freq]/xi_0
             ,  marker=Mar[num_mar%len(Mar)]
            #  ,  markevery=6000
         , color=colors[(num_mar//1)%len(colors)]
         ,  label=labs[y]+r'$Ste_{\mathrm{ref}}$, '+'TrioCFD'
        )
    num_mar = num_mar +1

num_mar = 0
# for y in range(nb_test) :
for y in range(2,5) :
# for y in [1] :
    
    time = np.linspace(0, tmax[y], 101)
    x_ana = x_position(time, beta[y], tshift[y])
    plt.plot(time/tshift[y]+1,x_ana/xi_0
             , color=colors[(num_mar//1)%len(colors)]
             ,  label=labs[y]+r'$Ste_{\mathrm{ref}}$, '+'Ana.'
            )
    
    num_mar = num_mar +1
    

plt.xlabel(r'$t/t_{\mathrm{ref}}$')
plt.ylabel(r"$x_i/x_{i, int}$")                  
plt.legend(loc="upper center", ncol=2, bbox_to_anchor=(0.5, 1.4))
plt.xlim(30, 40)
plt.ylim(5.4, 6.8)
# plt.title(r"$\frac{r}{R_0}$",r"$\frac{T_l-T_{sat}}{\Delta T_{\infty}}$")
# plt.savefig(f'{current_directory}/radius_tcl4bis.png', bbox_inches='tight')
plt.savefig(f'Effect_Ste_xi_ana_zoom.eps', bbox_inches='tight')
plt.show()

### Error of interface position at $74t_{\mathrm{ref}}$

In [ ]:
err = []
for y in range(2,5) :
    fname =f'M{y}'
    with open(f'build/{fname}/vol.txt', 'r') as file:
        last_line = file.readlines()[-1].strip()
        valeurs = last_line.split()
        vol = float(valeurs[1])
        time= float(valeurs[3])
        x_simu = np.asarray(vol)/W
        x_theric = x_position(tmax[y], beta[y], tshift[y])
        err.append(abs(x_simu-x_theric)/x_theric)




plt.scatter(np.asarray(dT[2:5])*Cp_v/h_lg, err, marker="o", label = f'present')
x = np.asarray(nombre_de_noeuds)
# plt.plot(x, 1/x**2, label='2nd order')  
# plt.plot(x, 0.03/x, label='1st order')

plt.xlabel(r"Ste")
plt.ylabel(r"Relative error")    
plt.xscale('log', base=2)
plt.yscale('log')
# plt.legend(loc="best")
# plt.title(r"$\frac{r}{R_0}$",r"$\frac{T_l-T_{sat}}{\Delta T_{\infty}}$")
# plt.savefig(f'{current_directory}/radius_tcl4bis.png', bbox_inches='tight')
plt.savefig(f'Error_Ste.eps', bbox_inches='tight')
plt.show()